# Hybrid + residual lifetime recalculation

This notebook recalculates the Yb lifetime checks with two additions beyond the repaired `get_nearby` workflow:

1. **resolved NIST supplement**: add low-lying experimentally tabulated NIST final states and replace overlapping low-energy MQDT approximants;
2. **unresolved residual sink**: where a literature total lifetime is available, add the remaining decay rate as a single effective loss channel.

The residual sink is reported separately. It is not a resolved physical final state; it is the rate required to make the finite explicit channel model match a benchmark total lifetime.


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import sys
import time
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-rydcalc")

from neutral_yb.external.rydcalc_adapter import build_yb171_atom, load_rydcalc  # noqa: E402

rydcalc = load_rydcalc(require_c_extension=True)
yb174 = rydcalc.Ytterbium174(cpp_numerov=True, use_db=False)
yb171 = build_yb171_atom(cpp_numerov=True, use_db=False, require_c_extension=True)
print(yb174.name, yb171.name)


174Yb 171Yb


## Literature reference values

Fang et al., JQSRT 69, 469-473 (2001), Table 1 reports room-temperature lifetimes for neutral Yb `6sns` and `6snd` states. The `fang_300K_us` column below is the paper's calculated room-temperature lifetime `1 / (A + B rho)`.

For the original `^171Yb |65 3S1, F=3/2, mF=-3/2>` target state, Muniz et al. report a measured Rydberg lifetime of `65(3) us` under typical tweezer conditions. That value contains experimental loss mechanisms beyond bare radiative decay, so the residual sink is an effective loss rate, not a resolved radiative branching channel.


In [2]:
FANG_2001 = {
    ("s", 21): {"fang_300K_us": 6.8, "fang_spont_us": 7.6},
    ("s", 22): {"fang_300K_us": 8.3, "fang_spont_us": 9.1},
    ("s", 23): {"fang_300K_us": 9.8, "fang_spont_us": 10.7},
    ("s", 24): {"fang_300K_us": 11.2, "fang_spont_us": 12.5},
    ("s", 25): {"fang_300K_us": 13.0, "fang_spont_us": 14.5},
    ("s", 26): {"fang_300K_us": 14.9, "fang_spont_us": 16.7},
    ("s", 27): {"fang_300K_us": 16.9, "fang_spont_us": 19.2},
    ("d", 21): {"fang_300K_us": 7.8, "fang_spont_us": 8.6},
    ("d", 22): {"fang_300K_us": 9.0, "fang_spont_us": 10.1},
    ("d", 23): {"fang_300K_us": 10.6, "fang_spont_us": 11.7},
    ("d", 24): {"fang_300K_us": 12.1, "fang_spont_us": 13.5},
    ("d", 25): {"fang_300K_us": 13.7, "fang_spont_us": 15.5},
    ("d", 26): {"fang_300K_us": 15.9, "fang_spont_us": 17.6},
    ("d", 27): {"fang_300K_us": 18.2, "fang_spont_us": 20.1},
}

YB171_TARGET_TAU_300K_US = 65.0


In [3]:
def state_signature(state, *, energy_bin_hz: float = 1e6):
    return (
        round(state.get_energy_Hz() / energy_bin_hz),
        round(2 * state.f),
        round(2 * state.m),
        int(state.parity),
    )


def get_channel_l(state):
    return int(round(state.channels[0].l))


def build_windowed_mqdt_candidates(
    atom,
    initial,
    *,
    nu_min: float = 3.0,
    nu_max: float | None = None,
    window_width: float = 2.0,
    window_overlap: float = 0.05,
    dl: int = 1,
    df: int = 1,
    dm: int = 1,
):
    if nu_max is None:
        nu_max = initial.nub + 10.0
    unique = {}
    raw_states = 0
    errors = []
    start = time.perf_counter()
    lo = float(nu_min)
    while lo < nu_max - 1e-12:
        hi = min(float(nu_max), lo + float(window_width))
        query_lo = lo - 0.5 * window_overlap
        query_hi = hi + 0.5 * window_overlap
        center = 0.5 * (query_lo + query_hi)
        half_width = 0.5 * (query_hi - query_lo)
        proxy = copy.copy(initial)
        proxy.nub = center
        proxy.v_exact = center
        proxy.n = center
        proxy.v = center
        try:
            states = atom.get_nearby(
                proxy,
                include_opts={"dn": half_width, "dl": dl, "df": df, "dm": dm},
            )
        except Exception as exc:
            states = []
            errors.append({"lo": query_lo, "hi": query_hi, "error": f"{type(exc).__name__}: {exc}"})
        raw_states += len(states)
        for final_state in states:
            if final_state is None:
                continue
            try:
                energy = float(final_state.get_energy_Hz())
                if not math.isfinite(energy):
                    continue
                if not initial.allowed_multipole(final_state, k=1):
                    continue
                key = state_signature(final_state)
            except Exception:
                continue
            unique.setdefault(
                key,
                {
                    "state": final_state,
                    "source": "MQDT",
                    "L": get_channel_l(final_state),
                    "energy_hz": energy,
                    "label": str(final_state),
                },
            )
        lo = hi
    return {
        "candidates": list(unique.values()),
        "raw_states": raw_states,
        "errors": errors,
        "seconds": time.perf_counter() - start,
    }


def load_nist_data(isotope: int):
    filename = f"Yb{isotope}_NIST.txt"
    return json.loads((root / "rydcalc" / "rydcalc" / filename).read_text(encoding="utf-8"))[1:]


def build_nist_candidates_yb174(atom, initial, *, same_spin_singlet: bool = True):
    initial_l = get_channel_l(initial)
    allowed_l = {ell for ell in (initial_l - 1, initial_l + 1) if ell >= 0}
    candidates = []
    max_energy_by_l = {}
    for item in load_nist_data(174):
        ell = int(item["l"])
        if ell not in allowed_l:
            continue
        if same_spin_singlet and int(item["S"]) != 0:
            continue
        for m_value in (initial.m - 1, initial.m, initial.m + 1):
            if abs(m_value) > item["J"]:
                continue
            final_state = atom.get_state((item["n"], item["S"], ell, item["J"], m_value), tt="NIST")
            if final_state is None:
                continue
            try:
                if not initial.allowed_multipole(final_state, k=1):
                    continue
                energy = float(final_state.get_energy_Hz())
            except Exception:
                continue
            candidates.append(
                {
                    "state": final_state,
                    "source": "NIST",
                    "L": ell,
                    "energy_hz": energy,
                    "label": f"NIST {item['n']} {'singlet' if item['S'] == 0 else 'triplet'} L={ell} J={item['J']} m={m_value:g}",
                }
            )
            max_energy_by_l[ell] = max(max_energy_by_l.get(ell, -math.inf), energy)
    return candidates, max_energy_by_l


def possible_f_values_yb171(j_value):
    # I = 1/2 for 171Yb.
    return sorted({abs(j_value - 0.5), j_value + 0.5})


def build_nist_candidates_yb171(atom, initial):
    initial_l = get_channel_l(initial)
    allowed_l = {ell for ell in (initial_l - 1, initial_l + 1) if ell >= 0}
    candidates = []
    max_energy_by_l = {}
    for item in load_nist_data(171):
        ell = int(item["l"])
        if ell not in allowed_l:
            continue
        for f_value in possible_f_values_yb171(float(item["J"])):
            for m_value in (initial.m - 1, initial.m, initial.m + 1):
                if abs(m_value) > f_value:
                    continue
                final_state = atom.get_state((item["n"], item["S"], ell, item["J"], f_value, m_value), tt="NIST")
                if final_state is None:
                    continue
                try:
                    if not initial.allowed_multipole(final_state, k=1):
                        continue
                    energy = float(final_state.get_energy_Hz())
                except Exception:
                    continue
                candidates.append(
                    {
                        "state": final_state,
                        "source": "NIST",
                        "L": ell,
                        "energy_hz": energy,
                        "label": f"NIST {item['n']} S={item['S']} L={ell} J={item['J']} F={f_value:g} m={m_value:g}",
                    }
                )
                max_energy_by_l[ell] = max(max_energy_by_l.get(ell, -math.inf), energy)
    return candidates, max_energy_by_l


def build_hybrid_candidates(atom, initial, *, isotope: int, window_width: float = 2.0, mqdt_nu_min: float = 3.0):
    mqdt = build_windowed_mqdt_candidates(atom, initial, nu_min=mqdt_nu_min, window_width=window_width)
    if isotope == 174:
        nist_candidates, max_energy_by_l = build_nist_candidates_yb174(atom, initial, same_spin_singlet=True)
    elif isotope == 171:
        nist_candidates, max_energy_by_l = build_nist_candidates_yb171(atom, initial)
    else:
        raise ValueError(isotope)

    pool = {}
    counts = {"NIST": 0, "MQDT": 0}
    skipped_low_mqdt = 0
    for candidate in nist_candidates:
        pool[state_signature(candidate["state"])] = candidate
        counts["NIST"] += 1

    for candidate in mqdt["candidates"]:
        ell = candidate["L"]
        # If NIST covers the low-energy part of this L manifold, use NIST there rather than the MQDT approximant.
        if ell in max_energy_by_l and candidate["energy_hz"] <= max_energy_by_l[ell] + 1e9:
            skipped_low_mqdt += 1
            continue
        key = state_signature(candidate["state"])
        if key not in pool:
            pool[key] = candidate
            counts["MQDT"] += 1

    return {
        "candidates": list(pool.values()),
        "mqdt_meta": mqdt,
        "counts": counts,
        "skipped_low_mqdt": skipped_low_mqdt,
        "nist_max_energy_by_l": max_energy_by_l,
    }


def decay_summary(atom, initial, candidates, *, temperature_k: float):
    env = rydcalc.environment(T_K=temperature_k)
    rows = []
    by_source = {"NIST": 0.0, "MQDT": 0.0}
    failures = 0
    start = time.perf_counter()
    for candidate in candidates:
        try:
            gamma = atom.partial_decay(initial, candidate["state"], env)
        except Exception:
            failures += 1
            continue
        if not math.isfinite(float(gamma)) or gamma == 0:
            continue
        gamma = float(gamma)
        by_source[candidate["source"]] = by_source.get(candidate["source"], 0.0) + gamma
        rows.append({**candidate, "gamma_s^-1": gamma})
    rows.sort(key=lambda row: row["gamma_s^-1"], reverse=True)
    gamma_total = sum(row["gamma_s^-1"] for row in rows)
    return {
        "gamma_s^-1": gamma_total,
        "tau_us": math.inf if gamma_total == 0 else 1e6 / gamma_total,
        "rows": rows,
        "by_source": by_source,
        "failures": failures,
        "seconds": time.perf_counter() - start,
    }


def add_residual_to_reference(summary, *, reference_tau_us: float):
    reference_gamma = 1e6 / reference_tau_us
    residual = max(0.0, reference_gamma - summary["gamma_s^-1"])
    corrected_gamma = summary["gamma_s^-1"] + residual
    return {
        "reference_tau_us": reference_tau_us,
        "reference_gamma_s^-1": reference_gamma,
        "residual_gamma_s^-1": residual,
        "corrected_gamma_s^-1": corrected_gamma,
        "corrected_tau_us": 1e6 / corrected_gamma,
    }


## Fang benchmark recomputation

For the Fang comparison, `Yb174` singlet states are used. The explicit hybrid model includes same-spin NIST final states plus MQDT final states above the NIST-covered low-energy region. The residual column is then the remaining rate required to match Fang's `1/(A+B rho)` room-temperature lifetime.


In [4]:
def fang_initial_state(series: str, n: int):
    if series == "s":
        return yb174.get_state((n, 0, 0, 0), tt="vlfm")
    if series == "d":
        return yb174.get_state((n, 2, 2, 0), tt="vlfm")
    raise ValueError(series)


fang_rows = []
for series in ("s", "d"):
    for n in range(21, 28):
        initial = fang_initial_state(series, n)
        hybrid = build_hybrid_candidates(yb174, initial, isotope=174, window_width=2.0)
        explicit_300 = decay_summary(yb174, initial, hybrid["candidates"], temperature_k=300.0)
        explicit_0 = decay_summary(yb174, initial, hybrid["candidates"], temperature_k=0.0)
        reference_tau = FANG_2001[(series, n)]["fang_300K_us"]
        residual = add_residual_to_reference(explicit_300, reference_tau_us=reference_tau)
        fang_rows.append(
            {
                "series": series,
                "n": n,
                "state": str(initial),
                "explicit_tau_300K_us": explicit_300["tau_us"],
                "explicit_gamma_300K_s^-1": explicit_300["gamma_s^-1"],
                "explicit_tau_0K_us": explicit_0["tau_us"],
                "fang_tau_300K_us": reference_tau,
                "residual_gamma_s^-1": residual["residual_gamma_s^-1"],
                "corrected_tau_us": residual["corrected_tau_us"],
                "n_candidates": len(hybrid["candidates"]),
                "counts": hybrid["counts"],
                "skipped_low_mqdt": hybrid["skipped_low_mqdt"],
                "by_source_300K": explicit_300["by_source"],
            }
        )
        print(
            f"{series}{n}: explicit={explicit_300['tau_us']:.3f} us, "
            f"Fang={reference_tau:.1f} us, residual={residual['residual_gamma_s^-1']:.3e} s^-1, "
            f"corrected={residual['corrected_tau_us']:.3f} us"
        )


s21: explicit=7.424 us, Fang=6.8 us, residual=1.235e+04 s^-1, corrected=6.800 us
s22: explicit=8.417 us, Fang=8.3 us, residual=1.671e+03 s^-1, corrected=8.300 us
s23: explicit=9.483 us, Fang=9.8 us, residual=0.000e+00 s^-1, corrected=9.483 us
s24: explicit=10.625 us, Fang=11.2 us, residual=0.000e+00 s^-1, corrected=10.625 us
s25: explicit=11.842 us, Fang=13.0 us, residual=0.000e+00 s^-1, corrected=11.842 us
s26: explicit=13.136 us, Fang=14.9 us, residual=0.000e+00 s^-1, corrected=13.136 us
s27: explicit=14.508 us, Fang=16.9 us, residual=0.000e+00 s^-1, corrected=14.508 us
d21: explicit=15.218 us, Fang=7.8 us, residual=6.249e+04 s^-1, corrected=7.800 us
d22: explicit=16.407 us, Fang=9.0 us, residual=5.016e+04 s^-1, corrected=9.000 us
d23: explicit=10.295 us, Fang=10.6 us, residual=0.000e+00 s^-1, corrected=10.295 us
d24: explicit=22.650 us, Fang=12.1 us, residual=3.849e+04 s^-1, corrected=12.100 us
d25: explicit=23.984 us, Fang=13.7 us, residual=3.130e+04 s^-1, corrected=13.700 us
d26: 

In [5]:
for series in ("s", "d"):
    print(f"Fang benchmark {series}-series")
    print(" n  explicit_300K  Fang_300K  residual_s^-1  corrected  cand  NIST/MQDT")
    for row in [row for row in fang_rows if row["series"] == series]:
        print(
            f"{row['n']:2d} "
            f"{row['explicit_tau_300K_us']:13.3f} "
            f"{row['fang_tau_300K_us']:10.1f} "
            f"{row['residual_gamma_s^-1']:13.3e} "
            f"{row['corrected_tau_us']:9.3f} "
            f"{row['n_candidates']:5d} "
            f"{row['counts']['NIST']}/{row['counts']['MQDT']}"
        )
    print()


Fang benchmark s-series
 n  explicit_300K  Fang_300K  residual_s^-1  corrected  cand  NIST/MQDT
21         7.424        6.8     1.235e+04     6.800   162 21/141
22         8.417        8.3     1.671e+03     8.300   168 21/147
23         9.483        9.8     0.000e+00     9.483   174 21/153
24        10.625       11.2     0.000e+00    10.625   180 21/159
25        11.842       13.0     0.000e+00    11.842   186 21/165
26        13.136       14.9     0.000e+00    13.136   192 21/171
27        14.508       16.9     0.000e+00    14.508   198 21/177

Fang benchmark d-series
 n  explicit_300K  Fang_300K  residual_s^-1  corrected  cand  NIST/MQDT
21        15.218        7.8     6.249e+04     7.800   513 21/492
22        16.407        9.0     5.016e+04     9.000   531 21/510
23        10.295       10.6     0.000e+00    10.295   549 21/528
24        22.650       12.1     3.849e+04    12.100   567 21/546
25        23.984       13.7     3.130e+04    13.700   585 21/564
26        25.923       15.9

## Original `^171Yb` target recomputation

The target state is

\[
|r\rangle = |65\,{}^3S_1, F=3/2, m_F=-3/2\rangle.
\]

The explicit hybrid model adds low-lying NIST P states to the windowed MQDT P manifold and uses NIST low P states plus only MQDT states with `nu >= 12`, avoiding low-`nu` MQDT/NIST overlap. At 300 K we also report the residual loss rate needed to match the measured `65 us` lifetime from Muniz et al.


In [6]:
target = yb171.get_state((65, 0, 1.5, -1.5), tt="vlfm")
print(target)
print(f"nub={target.nub:.8f}")

target_hybrid = build_hybrid_candidates(yb171, target, isotope=171, window_width=5.0, mqdt_nu_min=12.0)
target_0 = decay_summary(yb171, target, target_hybrid["candidates"], temperature_k=0.0)
target_300 = decay_summary(yb171, target, target_hybrid["candidates"], temperature_k=300.0)
target_residual_300 = add_residual_to_reference(target_300, reference_tau_us=YB171_TARGET_TAU_300K_US)

print("Yb171 target explicit hybrid (NIST low P + MQDT nu >= 12)")
print(f"  candidates        = {len(target_hybrid['candidates'])}")
print(f"  NIST/MQDT         = {target_hybrid['counts']['NIST']}/{target_hybrid['counts']['MQDT']}")
print(f"  skipped low MQDT  = {target_hybrid['skipped_low_mqdt']}")
print(f"  0 K gamma         = {target_0['gamma_s^-1']:.6e} s^-1")
print(f"  0 K tau           = {target_0['tau_us']:.3f} us")
print(f"  300 K gamma       = {target_300['gamma_s^-1']:.6e} s^-1")
print(f"  300 K tau         = {target_300['tau_us']:.3f} us")
print(f"  300 K by source   = {target_300['by_source']}")
print("Yb171 target residual-calibrated to 65 us")
print(f"  reference gamma   = {target_residual_300['reference_gamma_s^-1']:.6e} s^-1")
print(f"  residual gamma    = {target_residual_300['residual_gamma_s^-1']:.6e} s^-1")
print(f"  corrected tau     = {target_residual_300['corrected_tau_us']:.3f} us")


|171Yb:64.56,L=0,F=1.5,-1.5>
nub=64.56106363
Yb171 target explicit hybrid (NIST low P + MQDT nu >= 12)
  candidates        = 1542
  NIST/MQDT         = 84/1458
  skipped low MQDT  = 0
  0 K gamma         = 3.262577e+03 s^-1
  0 K tau           = 306.506 us
  300 K gamma       = 7.303500e+03 s^-1
  300 K tau         = 136.921 us
  300 K by source   = {'NIST': 3041.536641461625, 'MQDT': 4261.963388091779}
Yb171 target residual-calibrated to 65 us
  reference gamma   = 1.538462e+04 s^-1
  residual gamma    = 8.081115e+03 s^-1
  corrected tau     = 65.000 us


In [7]:
print("Top explicit target channels at 300 K")
for index, row in enumerate(target_300["rows"][:25], start=1):
    delta_ghz = (target.get_energy_Hz() - row["state"].get_energy_Hz()) / 1e9
    print(
        f"{index:2d}. gamma={row['gamma_s^-1']:.6e} s^-1, "
        f"delta={delta_ghz:.3f} GHz, {row['source']}, {row['label']}"
    )


Top explicit target channels at 300 K
 1. gamma=5.444279e+02 s^-1, delta=920555.090 GHz, NIST, NIST 6 S=1 L=1 J=2 F=2.5 m=-2.5
 2. gamma=5.104346e+02 s^-1, delta=-12.448 GHz, MQDT, |171Yb:65.08,L=1,F=2.5,-2.5>
 3. gamma=4.835891e+02 s^-1, delta=11.987 GHz, MQDT, |171Yb:64.08,L=1,F=2.5,-2.5>
 4. gamma=2.778149e+02 s^-1, delta=972073.534 GHz, NIST, NIST 6 S=1 L=1 J=1 F=1.5 m=-1.5
 5. gamma=2.568838e+02 s^-1, delta=-16.940 GHz, MQDT, |171Yb:65.27,L=1,F=1.5,-1.5>
 6. gamma=2.182922e+02 s^-1, delta=-18.591 GHz, MQDT, |171Yb:65.34,L=1,F=0.5,-0.5>
 7. gamma=2.177712e+02 s^-1, delta=920555.090 GHz, NIST, NIST 6 S=1 L=1 J=2 F=2.5 m=-1.5
 8. gamma=2.041738e+02 s^-1, delta=-12.448 GHz, MQDT, |171Yb:65.08,L=1,F=2.5,-1.5>
 9. gamma=1.934356e+02 s^-1, delta=11.987 GHz, MQDT, |171Yb:64.08,L=1,F=2.5,-1.5>
10. gamma=1.872883e+02 s^-1, delta=355702.548 GHz, NIST, NIST 7 S=1 L=1 J=2 F=2.5 m=-2.5
11. gamma=1.852241e+02 s^-1, delta=993165.972 GHz, NIST, NIST 6 S=1 L=1 J=0 F=0.5 m=-0.5
12. gamma=1.852099e+0

## Summary

The explicit hybrid model is the best current finite-channel calculation in this notebook: NIST low states plus windowed MQDT high states. The residual sink is then added only when matching a literature total lifetime is required.

For dynamics simulations, use explicit channels when the final-state branching matters. Use the residual sink only as population loss from the modeled Hilbert space.
